In [1]:
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import shap

c:\Users\Swayam Bhoir\Desktop\portfolio projects\Fraud Detection Engine with Explainable AI\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def generate_synthetic_transactions(n_samples=50000):
    np.random.seed(42)
    amount = np.random.exponential(scale=100, size=n_samples)
    distance_from_home = np.random.gamma(shape=2, scale=5, size=n_samples)
    time_since_last_tx = np.random.exponential(scale=30, size=n_samples)
    is_foreign = np.random.choice([0, 1], size=n_samples, p=[0.95, 0.05])
    failed_pin_attempts = np.random.choice([0, 1, 2, 3], size=n_samples, p=[0.85, 0.1, 0.04, 0.01])

    # Non-linear probability thresholding
    fraud_score = (
        0.003 * amount +
        0.04 * distance_from_home -
        0.01 * time_since_last_tx +
        1.5 * is_foreign +
        0.8 * failed_pin_attempts
    )

    fraud_prob = 1 / (1 + np.exp(-(fraud_score - 3)))
    is_fraud = (np.random.rand(n_samples) < fraud_prob).astype(int)

    return pd.DataFrame({
        "amount": amount,
        "distance_from_home": distance_from_home,
        "time_since_last_tx": time_since_last_tx,
        "is_foreign": is_foreign,
        "failed_pin_attempts": failed_pin_attempts,
        "is_fraud": is_fraud
    })

In [3]:
generate_synthetic_transactions()

,amount,distance_from_home,time_since_last_tx,is_foreign,failed_pin_attempts,is_fraud
0,46.926809,8.210884,4.156620,0,1,0
1,301.012143,18.808002,5.691330,0,0,0
2,131.674569,26.010783,69.672263,0,0,0
3,91.294255,1.831241,86.828094,0,0,0
4,16.962487,2.946890,8.336741,0,0,0
...,...,...,...,...,...,...
49995,97.403956,4.607953,2.960492,0,0,0
49996,25.643353,20.568924,28.695929,0,0,0
49997,138.228276,6.052486,45.304627,0,0,0
49998,73.607746,14.017068,106.038488,0,0,0


In [ ]:
def train_and_save():
    df = generate_synthetic_transactions()
    X = df.drop(columns=["is_fraud"])
    y = df["is_fraud"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    model = LGBMClassifier(
        n_estimators=150,
        learning_rate=0.05,
        scale_pos_weight=10,
        random_state=42,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    y_pred_proba = model.predict_proba(X_test)[:, 1]
    print(f"[+] Model Trained. ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

    explainer = shap.TreeExplainer(model)

    joblib.dump(model, "../fraud_model.pkl")
    joblib.dump(explainer, "../shap_explainer.pkl")
    print("[+] Model and Explainer artifacts saved to disk.")

In [12]:
train_and_save()

[[0.56881834 0.43118166]
 [0.42255805 0.57744195]
 [0.67597462 0.32402538]
 ...
 [0.60533575 0.39466425]
 [0.51906354 0.48093646]
 [0.54456053 0.45543947]]
[+] Model Trained. ROC-AUC Score: 0.6913
[+] Model and Explainer artifacts saved to disk.
